In [ ]:
%reload_ext autoreload
%autoreload 2
%matplotlib inline
from collections import defaultdict
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
import seaborn as sns
import pandas as pd
from pathlib import Path
import tbparse

EXPERIMENT_NAME = "compare_methods_squadv2_t5"
EXPERIMENT_DIR = Path("..") / "output" / EXPERIMENT_NAME
PLOT_DIR = Path("..") / "plots"
PLOT_DIR.mkdir(exist_ok=True)
PLOT_SUFFIX = ".pdf"

In [ ]:
def get_task_dfs(dedup="last"):
    task_dfs = defaultdict(list)
    for file in EXPERIMENT_DIR.glob("*"):
        run_dir = Path(file)
        if not run_dir.is_dir():
            continue  # skip non-directories
        if run_dir.name.startswith("."):
            continue  # skip hidden directories
        print(run_dir)

        df = tbparse.SummaryReader(run_dir, pivot=False).scalars
        if dedup == "first":
            df = df.drop_duplicates(subset=["step", "tag"], keep="first")
        elif dedup == "last":
            df = df.drop_duplicates(subset=["step", "tag"], keep="last")
        elif dedup == "mean":
            df = df.groupby(["step", "tag"], as_index=False)["value"].mean()
        elif dedup == "max":
            df = df.groupby(["step", "tag"], as_index=False)["value"].max()
        else:
            raise ValueError(f"Unknown dedup method: {dedup}")

        df = df.pivot(index="step", columns="tag", values="value").reset_index()

        keyvals = file.name.split(",")
        df["Seed"] = int(keyvals[0].replace("seed=", ""))
        # df["Task"] = keyvals[1].replace("task=", "")
        df["Method"] = keyvals[1].replace("method=", "")
        rank = float(keyvals[2].replace("rank=", ""))
        # df["Rank"] = float(keyvals[3].replace("rank=", ""))
        df["LR"] = float(keyvals[3].replace("lr=", ""))

        task_dfs[rank].append(df)

    return {task: pd.concat(dfs_list).reset_index() for task, dfs_list in task_dfs.items()}

task_df_dict = get_task_dfs()

In [ ]:
display(next(iter(task_df_dict.values())).columns.tolist())
for (rank, df) in task_df_dict.items():
    print(f"Rank: {rank}")

In [ ]:
experiment_df = task_df_dict[8]  # there is one experiment for now
experiment_df

In [ ]:
method_to_pretty = {
    "full": "Full",
    "lora": "LoRA",
    "svdlora": "SVDLoRA",
    "precond_lora": "RPLoRA",
    "oplora_proj": "Proj. PSI-LoRA",
    "oplora_scaled": "Scaled PSI-LoRA",
}

In [ ]:
# --- Inspect experiment_df schema (safe / lightweight) ---
import pandas as pd
import numpy as np

print("experiment_df shape:", experiment_df.shape)
print("columns:")
print(list(experiment_df.columns))

# show a compact preview
display(experiment_df.head(3))

# heuristic: find likely method column
candidate_method_cols = [c for c in ["method", "algo", "optimizer", "adapter", "adapter_method", "name"] if c in experiment_df.columns]
print("candidate method cols:", candidate_method_cols)

# heuristic: list-like columns (often store curves)
def _is_listlike(x):
    return isinstance(x, (list, tuple, np.ndarray, pd.Series))

listlike_cols = [c for c in experiment_df.columns if experiment_df[c].map(_is_listlike).any()]
print("list-like columns:", listlike_cols)

# numeric metric-ish columns
numeric_cols = [c for c in experiment_df.columns if pd.api.types.is_numeric_dtype(experiment_df[c])]
print("numeric columns (first 40):", numeric_cols[:40])
print("numeric columns count:", len(numeric_cols))


In [ ]:
# --- Plots + metrics table (pretty names, mean±std, LaTeX export) ---
from __future__ import annotations

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from IPython.display import display, Markdown

sns.set_theme(style="whitegrid", context="talk")

_df = experiment_df.copy()

# Normalize method naming
if "Method" not in _df.columns:
    raise KeyError("Expected a 'Method' column in experiment_df")

_df["MethodPretty"] = _df["Method"].map(method_to_pretty).fillna(_df["Method"]).astype(str)

# ------------------------
# Select best LR per method (using avg over seeds), then eval row per seed
# ------------------------
preferred_score_cols = [
    "eval/best_f1",
    "eval/f1",
    "eval/best_exact",
    "eval/exact",
]
score_col = next((c for c in preferred_score_cols if c in _df.columns), None)
if score_col is None:
    raise KeyError(f"Couldn't find any score column in {preferred_score_cols}")

eval_df = _df[_df[score_col].notna()].copy()
if eval_df.empty:
    raise ValueError(f"No eval rows found (all NaN in {score_col}).")

# Best checkpoint per (Method, Seed, LR)
keys_msl = ["Method", "MethodPretty", "Seed", "LR"]
idx_best_msl = eval_df.groupby(keys_msl, sort=False)[score_col].idxmax()
best_msl = eval_df.loc[idx_best_msl].copy()

# Pick best LR per method by averaging score across seeds
lr_scores = (
    best_msl.groupby(["Method", "MethodPretty", "LR"], sort=False)[score_col]
    .mean()
    .reset_index()
)
idx_best_lr = lr_scores.groupby(["Method", "MethodPretty"], sort=False)[score_col].idxmax()
selected_lr = lr_scores.loc[idx_best_lr, ["Method", "MethodPretty", "LR", score_col]].copy()
selected_lr = selected_lr.sort_values(["MethodPretty"]).reset_index(drop=True)

# Show best LR per method

def _fmt_lr(x: float) -> str:
    if pd.isna(x):
        return "--"
    x = float(x)
    # Prefer plain decimal for common LR ranges
    if 1e-4 <= abs(x) < 1e-1:
        return f"{x:.4g}"
    return f"{x:.2e}"


display(Markdown("### Best learning rate per method"))
display(
    selected_lr.assign(
        LR=selected_lr["LR"].map(_fmt_lr),
        **{f"Avg {score_col}": selected_lr[score_col]},
    )[["MethodPretty", "LR", f"Avg {score_col}"]]
    .rename(columns={"MethodPretty": "Method"})
)

# Keep (Method, Seed) rows for that single best LR per method
chosen = best_msl.merge(selected_lr[["Method", "MethodPretty", "LR"]], on=["Method", "MethodPretty", "LR"], how="inner").copy()

# ------------------------
# Training loss plot (only for chosen method-level best LR)
# ------------------------
train_df = _df[_df["train/loss"].notna()].copy()
if not train_df.empty:
    chosen_keys = chosen[["Method", "MethodPretty", "Seed", "LR"]].drop_duplicates()
    train_df = train_df.merge(chosen_keys, on=["Method", "MethodPretty", "Seed", "LR"], how="inner")

    plt.figure(figsize=(10.5, 6.0))
    ax = sns.lineplot(
        data=train_df.sort_values("step"),
        x="step",
        y="train/loss",
        hue="MethodPretty",
        estimator="mean",
        errorbar="sd",
        linewidth=2.0,
    )
    ax.set_title("Training loss (mean ± sd over seeds; best LR per method)")
    ax.set_xlabel("Step")
    ax.set_ylabel("Train loss")
    ax.legend(title="Method", bbox_to_anchor=(1.02, 1.0), loc="upper left")
    plt.tight_layout()
    plt.show()
else:
    display(Markdown("No training-loss rows found in `experiment_df` (column `train/loss` is all NaN)."))

# ------------------------
# Metrics table (mean±std across seeds; using method-level best LR)
# ------------------------
metric_cols_preferred = [
    # "eval/best_exact",
    # "eval/best_f1",
    "eval/exact",
    "eval/f1",
    "eval/loss",
    # "eval/best_exact_thresh",
    # "eval/best_f1_thresh",
]
metric_cols = [c for c in metric_cols_preferred if c in chosen.columns]
metric_cols = [c for c in metric_cols if pd.api.types.is_numeric_dtype(chosen[c])]

summary = chosen.groupby(["Method", "MethodPretty"], sort=False)[metric_cols].agg(["mean", "std"])

main_sort_col = score_col
if main_sort_col not in metric_cols:
    main_sort_col = metric_cols[0]

summary_sorted = summary.sort_values((main_sort_col, "mean"), ascending=False)
summary_sorted.columns = [f"{a}__{b}" for a, b in summary_sorted.columns]
summary_sorted = summary_sorted.reset_index(drop=False)

# Attach best LR per method for display/latex
lr_map = selected_lr.set_index("MethodPretty")["LR"].to_dict()
summary_sorted["BestLR"] = summary_sorted["MethodPretty"].map(lr_map)


def _fmt_num(x: float) -> str:
    if pd.isna(x):
        return "--"
    x = float(x)
    if abs(x) < 1.0:
        return f"{x:.4f}"
    if abs(x) < 100.0:
        return f"{x:.2f}"
    return f"{x:.1f}"


def _better_is_higher(col: str) -> bool:
    col_l = col.lower()
    if "loss" in col_l or "error" in col_l or "perplex" in col_l:
        return False
    return True


pretty_col_names = {
    "eval/best_exact": "Best EM",
    "eval/best_f1": "Best F1",
    "eval/exact": "EM",
    "eval/f1": "F1",
    "eval/loss": "Eval loss",
    "eval/best_exact_thresh": "Best EM thr",
    "eval/best_f1_thresh": "Best F1 thr",
}

rows = []
for _, r in summary_sorted.iterrows():
    row = {"Method": r["MethodPretty"], "Best LR": _fmt_lr(r["BestLR"])}
    for m in metric_cols:
        mean_v = r.get(f"{m}__mean", np.nan)
        std_v = r.get(f"{m}__std", np.nan)
        mean_s = _fmt_num(mean_v)
        std_s = _fmt_num(std_v)
        row[pretty_col_names.get(m, m)] = f"{mean_s} <span style='font-size: 75%;'>(±{std_s})</span>"
    rows.append(row)

display_df = pd.DataFrame(rows)
metric_display_cols = [pretty_col_names.get(m, m) for m in metric_cols]

means_for_rank = {
    pretty_col_names.get(m, m): summary_sorted[f"{m}__mean"].to_numpy()
    for m in metric_cols
}

best_idx = {}
second_idx = {}
for m in metric_cols:
    col_disp = pretty_col_names.get(m, m)
    vals = means_for_rank[col_disp]
    if _better_is_higher(m):
        order = np.argsort(-np.nan_to_num(vals, nan=-np.inf))
    else:
        order = np.argsort(np.nan_to_num(vals, nan=np.inf))
    order = [i for i in order if np.isfinite(vals[i])]
    best_idx[col_disp] = order[0] if len(order) > 0 else None
    second_idx[col_disp] = order[1] if len(order) > 1 else None


def _style_cell(val: str, row_i: int, col_name: str) -> str:
    if col_name in best_idx and best_idx[col_name] == row_i:
        return f"<b>{val}</b>"
    if col_name in second_idx and second_idx[col_name] == row_i:
        return f"<u>{val}</u>"
    return val


styled = display_df.copy()
for col in metric_display_cols:
    styled[col] = [_style_cell(styled.loc[i, col], i, col) for i in range(len(styled))]

display(Markdown("### Metrics (mean ± std over seeds; best LR per method)"))
display(styled.style.hide(axis="index").set_properties(**{"text-align": "left"}))

# ------------------------
# LaTeX table export (best bold, second underline; std in \scriptsize)
# ------------------------

def _latex_escape(s: str) -> str:
    return (
        s.replace("\\", "\\\\")
        .replace("_", "\\_")
        .replace("%", "\\%")
        .replace("&", "\\&")
        .replace("#", "\\#")
        .replace("$", "\\$")
        .replace("{", "\\{")
        .replace("}", "\\}")
        .replace("~", "\\textasciitilde{}")
        .replace("^", "\\textasciicircum{}")
    )


def _fmt_mean_std_latex(mean_v: float, std_v: float) -> str:
    mean_s = _fmt_num(mean_v)
    std_s = _fmt_num(std_v)
    if std_s == "--":
        return mean_s
    return f"{mean_s} {{\\scriptsize $\\pm$ {std_s}}}"


latex_rows = []
latex_metric_cols = metric_cols
latex_headers = [pretty_col_names.get(m, m) for m in latex_metric_cols]

for row_i, r in summary_sorted.iterrows():
    row_cells = [
        _latex_escape(str(r["MethodPretty"])),
        _latex_escape(_fmt_lr(r["BestLR"])),
    ]
    for m in latex_metric_cols:
        mean_v = float(r[f"{m}__mean"]) if pd.notna(r[f"{m}__mean"]) else np.nan
        std_v = float(r[f"{m}__std"]) if pd.notna(r[f"{m}__std"]) else np.nan
        cell = _fmt_mean_std_latex(mean_v, std_v)

        col_disp = pretty_col_names.get(m, m)
        if best_idx.get(col_disp, None) == row_i:
            cell = f"\\textbf{{{cell}}}"
        elif second_idx.get(col_disp, None) == row_i:
            cell = f"\\underline{{{cell}}}"

        row_cells.append(cell)

    latex_rows.append(" & ".join(row_cells) + r" \\")

col_spec = "l" + "c" + "c" * len(latex_metric_cols)
latex_table = "\n".join(
    [
        "\\begin{tabular}{" + col_spec + "}",
        "\\toprule",
        "Method & LR & " + " & ".join([_latex_escape(h) for h in latex_headers]) + r" \\",
        "\\midrule",
        *latex_rows,
        "\\bottomrule",
        "\\end{tabular}",
    ]
)

print("\nLaTeX table:\n")
print(latex_table)
